In [ ]:
# clear all the parameters
dbutils.widgets.removeAll()

In [ ]:
%run "../utils/setup_databricks_session"

In [ ]:
from pyspark.sql import functions as F

from core.configs import WIDGETS_CONFIG
from core.currency import get_currency_dim, get_currency_exchange_rate_fact
from core.engine import scoped_merge_engine
from core.databricks_session import create_required_widgets, fetch_widget_values, log, view

In [ ]:
# define the required widgets
required_widgets = [
    "execution_mode",
    "bronze_catalog",
    "bronze_schema_currency",
    "silver_catalog",
    "silver_schema",
    "target_currency_exchange_rate_fact",
    "target_currency_dim",
]

# Create widgets for required parameters
create_required_widgets(dbutils, WIDGETS_CONFIG, required_widgets)

In [ ]:
# Get widget values
params = fetch_widget_values(dbutils, print_validation=False)

# Define other parameters
params.exchange_rate_type_cd = F.col("KURST") == "M"  # daily exchange rate
params.from_currency_cd = F.col("FCURR").isin("CAD")
params.to_currency_cd = F.col("TCURR").isin("USD")
log(f"Execution Mode: {params.execution_mode}")

In [ ]:
tcurc_raw = spark.table(f"{params.bronze_catalog}.{params.bronze_schema_currency}.TCURC")
tcurt_raw = spark.table(f"{params.bronze_catalog}.{params.bronze_schema_currency}.TCURT")
tcurr_raw = spark.table(f"{params.bronze_catalog}.{params.bronze_schema_currency}.TCURR")

In [ ]:
currency_dim_df = get_currency_dim(tcurc_raw, tcurt_raw)
currency_exchange_rate_fact_df = get_currency_exchange_rate_fact(
    tcurr_raw, params.exchange_rate_type_cd, params.from_currency_cd, params.to_currency_cd
)
view(currency_dim_df, params.execution_mode)
view(currency_exchange_rate_fact_df, params.execution_mode)

In [ ]:
source = currency_dim_df
target = f"{params.silver_catalog}.{params.silver_schema}.{params.target_currency_dim}"
merge_key = "currency_ck"
write_mode = "reload"
scope = "1=1"

scoped_merge_engine(spark, source, target, merge_key, write_mode, scope)

In [ ]:
source = currency_exchange_rate_fact_df
target = f"{params.silver_catalog}.{params.silver_schema}.{params.target_currency_exchange_rate_fact}"
merge_key = "currency_exchange_rate_ck"
write_mode = "reload"
scope = "1=1"

scoped_merge_engine(spark, source, target, merge_key, write_mode, scope)